In [ ]:
!pip install --upgrade numpy
!pip install --upgrade --force-reinstall irrCAC

## Gwet AC1 Analysis of Final Study

In [ ]:
import pandas as pd
from irrCAC.raw import CAC
from irrCAC.benchmark import Benchmark
import csv

# File paths
input_csv_path = 'Gwet_Input_Table.csv'
output_csv_path = 'Gwet_AC1_values.csv'

# Metrics and raters
metrics = ['clarity', 'clinical_relevance', 'difficulty', 'option_accuracy', 'assessment_accuracy', 'feedback_quality']
raters = ['MP', 'VG', 'BS']

# Read input data
data = pd.read_csv(input_csv_path)

# Output fields
fieldnames = ['metric', 'coefficient_value', 'confidence_interval_ll', 'confidence_interval_ul', 'p_value', 'z', 'se', 'pa', 'pe', 'altman_interpretation']
output_rows = []

for metric in metrics:
    rater_cols = [f'{metric}_{r}' for r in raters]
    rater_data = data[rater_cols]

    # Compute Gwet's AC1
    cac = CAC(rater_data)
    gwet_result = cac.gwet()

    est = gwet_result['est']
    coeff = est['coefficient_value']
    se = est['se']

    # Determine Altman's interpretation
    benchmark = Benchmark(coeff=coeff, se=se)
    altman_info = benchmark.altman()

    altman_interpretation = "Unknown"
    for (low, high), label in zip(altman_info['scale'], altman_info['Altman']):
        if low <= coeff < high or (coeff == 1.0 and high == 1.0):  # Include upper bound only if it's 1.0
            altman_interpretation = label
            break

    row = {
        'metric': metric,
        'coefficient_value': coeff,
        'confidence_interval_ll': est['confidence_interval'][0],
        'confidence_interval_ul': est['confidence_interval'][1],
        'p_value': est['p_value'],
        'z': est['z'],
        'se': se,
        'pa': est['pa'],
        'pe': est['pe'],
        'altman_interpretation': altman_interpretation
    }

    output_rows.append(row)

# Save to CSV
df = pd.DataFrame(output_rows)
df.to_csv(output_csv_path, index=False)

# Display the result
print(df)
print(f"Gwet's AC1 values with Altman interpretation saved to {output_csv_path}")
